# A minimal GPT, character-level, trained on a CPU

This notebook builds and trains a complete GPT in PyTorch, with the attention written by hand (no `nn.Transformer`, no `nn.MultiheadAttention`) and every tensor shape annotated. Training takes about 10 seconds on a laptop CPU.

```text
tokens -> tok_emb + pos_emb -> [Block x 2] -> LayerNorm -> Linear -> logits
Block:  x = x + attn(LN(x));  x = x + mlp(LN(x))      (pre-LN, residual)
```

It matches these sections of notes.md:

- §3.5 (positional encodings): a learned absolute position table, as in GPT-2.
- §3.6 (the transformer block): pre-LN blocks with residual connections and a 4× MLP.
- §3.7 (causal masking): the mask, plus a check that changing future tokens leaves earlier outputs unchanged.
- §5.1 (next-token prediction): cross-entropy over the vocabulary at every position, with initial loss near $\ln V$.
- §5.4 (sampling): temperature sampling from the trained model.

Run: `uv run jupyter execute ml/04_deep_learning/mini_gpt.ipynb`

## Setup

`torch.manual_seed(42)` fixes the weight initialization, the sampled batches and the generated text. The cells below consume the global random stream in the same order on every run (one check in section 4 uses a forked stream, so it does not shift anything), which makes the printed losses and samples reproducible.

In [1]:
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

_ = torch.manual_seed(42)

## 1. The corpus

A few paragraphs of original prose, written for this notebook, so nothing is downloaded. The text is 1,616 characters long. A character-level model can learn spelling, spacing and common short words from this little text, and a model of this size will also memorize much of it.

In [2]:
CORPUS = """
The old miller kept a small grey cat, and the cat kept the mill. Every
morning the miller ground his grain, and every morning the cat walked the
beams above him, watching the wheat dust drift like slow snow. The miller
said the cat earned no bread, for it did no work. The cat said nothing at
all, for it was a cat, and the mice said nothing either, for they had long
ago moved to the baker's house across the river.

One winter the river froze and the wheel stood still. The miller counted
his sacks and found them full, though he had not ground grain in a month.
He counted again by candlelight, and a third time at dawn. The sacks were
full because nothing had touched them, and nothing had touched them because
the small grey cat still walked the beams, silent as the snow on the roof.

The miller was not a quick man, but he was not a fool either. He set a
bowl of warm milk by the door and never again spoke of bread, or of work,
or of what the cat had earned. Some wages are paid in what does not happen.
The wheel turned in spring, the grain flowed, and the cat grew old and
grey and fat, and slept in the sun like a stone that had decided to be
soft.

A young man from the town once asked the miller the secret of his tidy
mill. The miller pointed at the sleeping cat. The young man laughed and
said a sleeping cat does nothing. The miller nodded and poured the milk.
It is hard to teach a man the value of nothing happening, and the miller
did not try. He ground his grain. The wheel turned. The cat slept on, and
the mice stayed across the river, and that was the whole of the secret,
and it was enough.
"""

## 2. Character-level tokenizer

The vocabulary is the sorted set of distinct characters in the corpus, so `vocab_size` is the number of distinct characters (expected 36). `encode` maps a string to a list of integer ids and `decode` maps ids back to a string; the assert checks that decoding an encoded string returns it unchanged. BPE (notes §5.2) uses the same interface with multi-character units.

In [3]:
chars = sorted(set(CORPUS))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}   # char -> int
itos = {i: ch for ch, i in stoi.items()}       # int -> char
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[int(i)] for i in ids)

data = torch.tensor(encode(CORPUS), dtype=torch.long)   # (len_corpus,)
print(f"corpus: {len(CORPUS)} chars, vocab_size = {vocab_size}")
print(f"vocab: {''.join(chars)!r}")
assert decode(encode("The miller")) == "The miller"
print("encode('The miller') =", encode("The miller"))

corpus: 1616 chars, vocab_size = 36
vocab: "\n ',.AEHIOSTabcdefghiklmnopqrstuvwyz"
encode('The miller') = [11, 19, 16, 1, 23, 20, 22, 22, 16, 28]


## 3. Hyperparameters and training batches

The model is small on purpose: context length 64, width 64, 4 heads of width 16, 2 blocks, and 400 AdamW steps on batches of 32. `get_batch` samples random 64-character windows of the corpus; the targets `y` are the inputs `x` shifted left by one character, so every position is a training example (predict character $t+1$ from characters $\le t$). The cell prints the first 40 characters of one input-target pair taken from the start of the corpus (without using the random stream) so the one-character shift is visible.

In [4]:
block_size = 64    # max context length T (positions 0..63)
n_embd = 64        # embedding width d_model
n_head = 4         # attention heads; head_dim = 64/4 = 16
n_layer = 2        # transformer blocks
batch_size = 32
max_steps = 400
lr = 3e-3


def get_batch():
    # Sample batch_size random crops of the corpus.
    # x: (B, T) input chars;  y: (B, T) = x shifted left by one.
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])          # (B, T)
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])  # (B, T)
    return x, y


x0 = data[0:block_size]           # an input window starting at character 0
y0 = data[1:block_size + 1]       # its targets: the same window shifted by one
print("input  x:", repr(decode(x0[:40])))
print("target y:", repr(decode(y0[:40])))
assert torch.equal(x0[1:], y0[:-1])

input  x: '\nThe old miller kept a small grey cat, a'
target y: 'The old miller kept a small grey cat, an'


## 4. Causal self-attention, MLP and the transformer block

`CausalSelfAttention` computes $Q, K, V$ for all heads with one fused linear layer, splits them into heads, computes scores scaled by $1/\sqrt{d_{\text{head}}}$, sets scores for future positions to $-\infty$, applies softmax, averages the values, and merges the heads. `MLP` expands the width 4×, applies GELU and projects back; it acts on each position separately. `Block` is the pre-LN block of notes §3.6.

The check at the end builds one attention layer on a forked random stream, runs it on a random input, then changes the input at positions 10 to 19 only. With a correct causal mask, outputs at positions 0 to 9 are identical (expected maximum difference exactly 0) and outputs from position 10 on change.

In [5]:
class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.head_dim = n_embd // n_head
        # One fused linear computes Q, K, V for all heads at once:
        # (B,T,n_embd) -> (B,T,3*n_embd), then split.
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)   # output projection W_o
        # Causal mask: lower-triangular (T,T) of ones. register_buffer stores
        # it with the module but not as a parameter (no gradients).
        self.register_buffer(
            "mask", torch.tril(torch.ones(block_size, block_size))
        )

    def forward(self, x):
        B, T, C = x.shape                       # C = n_embd
        qkv = self.qkv(x)                       # (B, T, 3C)
        q, k, v = qkv.split(n_embd, dim=2)      # each (B, T, C)
        # Split channels into heads: (B, T, C) -> (B, n_head, T, head_dim)
        q = q.view(B, T, n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, n_head, self.head_dim).transpose(1, 2)

        # Attention scores: (B,h,T,d) @ (B,h,d,T) -> (B,h,T,T)
        # Scale by 1/sqrt(head_dim) so scores have ~unit variance at init
        # (Var(<q,k>) = head_dim for unit-variance entries).
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        # Causal mask: position i may not read j > i. -inf -> softmax weight 0.
        att = att.masked_fill(self.mask[:T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)            # rows sum to 1: (B,h,T,T)
        y = att @ v                             # (B,h,T,T)@(B,h,T,d)->(B,h,T,d)
        # Merge heads back: (B,h,T,d) -> (B,T,h,d) -> (B,T,C)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)                     # (B, T, C)


class MLP(nn.Module):
    # Position-wise feed-forward: expand 4x, nonlinearity, project back.
    # It acts on each position separately; attention is the only layer that
    # combines information across positions.
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(n_embd, 4 * n_embd)   # (C) -> (4C)
        self.proj = nn.Linear(4 * n_embd, n_embd) # (4C) -> (C)

    def forward(self, x):                          # (B, T, C) -> (B, T, C)
        return self.proj(F.gelu(self.fc(x)))


class Block(nn.Module):
    # Pre-LN transformer block: normalize, transform, add back (residual).
    # The residual additions give the gradient an identity path through
    # every block.
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention()
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = MLP()

    def forward(self, x):                          # (B, T, C)
        x = x + self.attn(self.ln1(x))             # mix information across positions
        x = x + self.mlp(self.ln2(x))              # per-position MLP
        return x


# Causality check on a forked random stream (restores the global state after).
with torch.random.fork_rng():
    torch.manual_seed(0)
    attn = CausalSelfAttention()
    xa = torch.randn(1, 20, n_embd)
    xb = xa.clone()
    xb[:, 10:] = torch.randn(1, 10, n_embd)       # change positions 10..19 only
    with torch.no_grad():
        ya, yb = attn(xa), attn(xb)
past = (ya[:, :10] - yb[:, :10]).abs().max().item()
future = (ya[:, 10:] - yb[:, 10:]).abs().max().item()
print(f"max change at positions 0-9:   {past:.2e}")
print(f"max change at positions 10-19: {future:.2e}")
assert past == 0.0 and future > 0.0

max change at positions 0-9:   0.00e+00
max change at positions 10-19: 2.86e-01


## 5. The full model

`MiniGPT` adds a token embedding (a $V \times C$ lookup table, notes §4.1) to a learned position embedding (a $T_{\max} \times C$ table), runs the blocks, applies a final LayerNorm, and maps each position to $V$ logits. With targets it returns the cross-entropy averaged over all $B \cdot T$ positions. `generate` samples one character at a time from the softmax of the last position's logits divided by the temperature, appends it, and repeats.

The cell creates the model and computes the loss on a fixed batch of 32 windows from the corpus (no random sampling). With random initial weights the predicted distribution is close to uniform, so the expected loss is near $\ln V = \ln 36 \approx 3.58$; a value somewhat above that is normal because the initial logits are not exactly equal.

In [6]:
class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd)   # (V, C) lookup table
        self.pos_emb = nn.Embedding(block_size, n_embd)   # (T_max, C) learned
        self.blocks = nn.ModuleList([Block() for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size)          # (C) -> (V) logits

    def forward(self, idx, targets=None):
        # idx: (B, T) integer char ids
        B, T = idx.shape
        tok = self.tok_emb(idx)                            # (B, T, C)
        pos = self.pos_emb(torch.arange(T))                # (T, C)
        x = tok + pos                                      # broadcast -> (B,T,C)
        # Without pos_emb, attention is permutation-equivariant: it could
        # not tell "cat sat" from "sat cat".
        for block in self.blocks:
            x = block(x)                                   # (B, T, C)
        x = self.ln_f(x)                                   # (B, T, C)
        logits = self.head(x)                              # (B, T, V)

        loss = None
        if targets is not None:
            # Cross-entropy over all B*T positions at once: V-way
            # classification of the next character.
            loss = F.cross_entropy(
                logits.view(B * T, vocab_size), targets.view(B * T)
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        # idx: (B, T0) prompt. Autoregressive loop: sample, append, repeat.
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]      # crop to context window
            logits, _ = self(idx_cond)           # (B, T, V)
            logits = logits[:, -1, :] / temperature   # (B, V): last position
            probs = F.softmax(logits, dim=-1)    # (B, V)
            nxt = torch.multinomial(probs, 1)    # (B, 1) sample
            idx = torch.cat([idx, nxt], dim=1)   # (B, T+1)
        return idx


model = MiniGPT()
n_params = sum(p.numel() for p in model.parameters())
print(f"model parameters: {n_params:,}")
# Expected loss at init: uniform over vocab -> -ln(1/V) = ln(V).
print(f"expected initial loss ~ ln({vocab_size}) = {math.log(vocab_size):.3f}")

# A fixed batch: 32 evenly spaced windows (deterministic, no random stream used).
starts = torch.linspace(0, len(data) - block_size - 1, batch_size).long()
x_fix = torch.stack([data[i:i + block_size] for i in starts])
y_fix = torch.stack([data[i + 1:i + block_size + 1] for i in starts])
with torch.no_grad():
    _, init_loss = model(x_fix, y_fix)
print(f"loss of the untrained model on a fixed batch: {init_loss.item():.3f}")
assert abs(init_loss.item() - math.log(vocab_size)) < 0.5

model parameters: 108,836
expected initial loss ~ ln(36) = 3.584
loss of the untrained model on a fixed batch: 3.754


## 6. Sample before training

Generate 250 characters starting from the newline character (id 0) at temperature 0.8. The weights are random, so the output should be an unstructured mix of the 36 characters, with no recognizable words.

In [7]:
def sample_text(model, n=250):
    start = torch.zeros((1, 1), dtype=torch.long)   # start from token 0 ('\n')
    out = model.generate(start, n, temperature=0.8)
    return decode(out[0].tolist())


print("--- sample BEFORE training ---")
print(sample_text(model))
print("------------------------------")

--- sample BEFORE training ---



OybhSkvrnEqqnI.yuc,Sytmh IcspoISOIIn k.o,iTvqiAopiod.'Aa bE
tub
dyAOzwwvwvnw I HE lyhHtEI IHcS kzSb vqrIb'.TuoncvuOznn'poqyw'sovApvAOe '.hwv,HI.
wvaOvyzps,etyk,ucwOsfbzsEiiwkufwOlTa dbI
I
kcAspoqwt
ltcSbSbev'g s,os.w,a'ndfwOmutSgHSzb
vgfwdpsecwOI Tun
------------------------------


## 7. Train

Run 400 AdamW steps at learning rate $3 \times 10^{-3}$, printing the batch loss every 50 steps. Expected: the loss starts near 3.7 (close to $\ln 36$), falls below 2 within about 100 steps as the model learns character frequencies and common pairs, and ends far lower (around 0.15 on this run) because the model memorizes much of the 1,616-character corpus. The assert requires the final loss to be below 60% of the first.

In [8]:
opt = torch.optim.AdamW(model.parameters(), lr=lr)
t0 = time.time()
first_loss = None
for step in range(max_steps):
    x, y = get_batch()                 # (B,T), (B,T)
    _, loss = model(x, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if first_loss is None:
        first_loss = loss.item()
    if step % 50 == 0 or step == max_steps - 1:
        print(f"step {step:4d}  loss {loss.item():.4f}")
dt = time.time() - t0
print(f"\ntrained {max_steps} steps in {dt:.1f}s "
      f"({dt / max_steps * 1000:.0f} ms/step)")
assert loss.item() < 0.6 * first_loss, "loss did not decrease enough"
print(f"initial loss {first_loss:.3f} vs ln({vocab_size}) = {math.log(vocab_size):.3f}")

step    0  loss 3.7630


step   50  loss 2.0442


step  100  loss 1.8244


step  150  loss 1.4147


step  200  loss 0.9053


step  250  loss 0.4185


step  300  loss 0.2340


step  350  loss 0.1999


step  399  loss 0.1557

trained 400 steps in 12.9s (32 ms/step)
initial loss 3.763 vs ln(36) = 3.584


## 8. Sample after training

Generate again with the same settings. Expected: correctly spelled words, spaces and punctuation in the right places, and long stretches copied from the corpus mixed with a few invented word combinations. The copying is memorization: about 109K parameters trained to a loss near 0.15 on a 1,616-character text.

In [9]:
print("--- sample AFTER training ---")
print(sample_text(model))
print("-----------------------------")

--- sample AFTER training ---

The miller the secret of his tidy
mill. The miller pointed at the sleeping cat doing nothing eithed, at dit did tn the suney cat dain spounte cat had not his tid to what a cat tond  the baker's rion, as the wer's whole as the secret of hidy
mill. Eve
-----------------------------
